In [1]:
import os
from pathlib import Path
import sys

# Automatically find repo root by looking for .git
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# Change the working directory to the repo root
os.chdir(ROOT)

In [2]:
import pandas as pd

In [3]:
from src.utils.data_loaders.read_settings_json import read_settings_json

args = read_settings_json()
args

{'Config': {'debug_mode': 'False', 'TEMP_CACHE': 'data/temp_cache'},
 'TrainingInput': {'CHART_OF_ACCOUNTS': 'data/training_input/chart_of_accounts.xlsx',
  'ENROLLEES': 'data/training_input/enrollees_pseudonymized.xlsx',
  'REVENUES': 'data/training_input/revenues_pseudonymized.xlsx'},
 'Training': {'MODEL_PARAMETERS': 'src/modules/machine_learning/parameters.json',
  'RESULTS_ROOT': 'data/training_results',
  'LOGS': 'data/training_logs',
  'DEPLOYED_MODELS': 'data/training_results/deployed_models',
  'observation_end': '2026/05/09',
  'target_feature': 'dtp_bracket',
  'test_size': '0.30'}}

In [4]:
df_revenues = pd.read_excel(args['TrainingInput']['REVENUES'], engine='calamine')

In [5]:
df_enrollees = pd.read_excel(args['TrainingInput']['ENROLLEES'], engine='calamine')

In [6]:
from src.modules.feature_engineering.credit_sales_machine_learning import CreditSalesProcessor

cs_test = CreditSalesProcessor(df_revenues, df_enrollees, args,
                      drop_fully_paid_invoices=True,
                      drop_back_account_transactions=True,
                      calculate_payment_amounts=True,
                      add_description=True,
                      drop_missing_dtp=False)
df_cs_test = cs_test.show_data()
df_cs_test

Single due date records:   11151
Multiple due date records: 289


Dropped 9896 fully paid invoices. Remaining: 511


,school_year,student_id_pseudonimized,category_name,gross_receivables,amount_discounted,adjustments,credit_sale_amount,due_date,date_fully_paid,prepayments,...,due_quarter,opening_balance_flag,payment_ratio,early_payer_flag,on_time_streak,prev_bracket,dtp_rolling_std,dtp_max,plan_type_risk_score,description
2464,2022,02PNVPI5,Kn2-C-2nd,2800.0,0.0,0.0,2800.0,2022-11-05,NaT,0.0,...,4,1,0.943797,1.0,1,0.0,<NA>,-3,0,Tuition fee (Kn2) - 2 of 4 payments
2930,2022,02PNVPI5,Kn2-OF-2nd,2267.0,0.0,0.0,2267.0,2022-12-05,NaT,0.0,...,4,1,0.881408,NaN,0,NaN,<NA>,-3,0,Miscellaneous fees - 2 of 3 payments
2971,2022,02PNVPI5,Events - Foundation Day,490.0,0.0,0.0,490.0,2022-12-16,NaT,0.0,...,4,1,0.805478,NaN,0,NaN,<NA>,-3,0,Foundation Day
2972,2022,02PNVPI5,Kn2 - Moving Up - Male,2600.0,0.0,0.0,2600.0,2022-12-16,NaT,0.0,...,4,1,0.805478,NaN,0,NaN,<NA>,-3,0,Moving up fee
2973,2022,02PNVPI5,Surcharge,336.0,0.0,0.0,336.0,2022-12-16,NaT,0.0,...,4,1,0.805478,NaN,0,NaN,<NA>,<NA>,0,Surcharge
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5256,2023,YRVVE1D9,Surcharge,117.0,0.0,0.0,117.0,2023-12-14,NaT,0.0,...,4,1,0.643593,NaN,0,NaN,<NA>,<NA>,0,Surcharge
5446,2023,YRVVE1D9,Kn2-C-4th,3900.0,0.0,0.0,3900.0,2024-02-09,NaT,0.0,...,1,1,0.559412,NaN,0,NaN,<NA>,<NA>,0,Tuition fee (Kn2) - 4 of 4 payments
5558,2023,YRVVE1D9,Kn2-Books,4300.0,0.0,0.0,4300.0,2024-03-08,NaT,0.0,...,1,1,0.425543,NaN,0,NaN,<NA>,<NA>,0,Books (Kn2)
5559,2023,YRVVE1D9,Kn2-OF-3rd,3600.0,0.0,0.0,3600.0,2024-03-08,NaT,0.0,...,1,1,0.425543,NaN,0,NaN,<NA>,<NA>,0,Miscellaneous fees - 3 of 3 payments


In [7]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")

from src.modules.machine_learning.utils.inference.inference_pipeline import (
    find_deployed_model,
    load_inference_pipeline,
    run_batch_inference,
)

MODEL_DIR = args["Training"]["DEPLOYED_MODELS"]

In [8]:
# Locate and inspect the deployed artifact.
# Raises ValueError with upgrade instructions if artifact is pre-InferencePipeline.
artifact_path = find_deployed_model(MODEL_DIR)
print("Artifact:", artifact_path)

try:
    pipeline = load_inference_pipeline(MODEL_DIR)
    print(pipeline)
except ValueError as e:
    msg = str(e)
    print("[WARN] Could not load InferencePipeline:")
    print(" ", msg)
    print()
    print("Action required: Re-run Step 5 (Model Finalization) in the app to regenerate the artifact.")
    pipeline = None


INFO src.modules.machine_learning.utils.inference.inference_pipeline Loading InferencePipeline from data\training_results\deployed_models\finalized_two_stage_xgb_ada.pkl


Artifact: data\training_results\deployed_models\finalized_two_stage_xgb_ada.pkl


[WARN] Could not load InferencePipeline:
  The artifact at data\training_results\deployed_models\finalized_two_stage_xgb_ada.pkl is in the pre-InferencePipeline flat-dict format (keys: ['pipeline', 'parameters', 'features', 'label_encoder']). It was saved before InferencePipeline became the deployment format.  To fix: open the app, go to Step 5 (Model Finalization), and re-deploy the model to regenerate a compatible artifact, then retry.

Action required: Re-run Step 5 (Model Finalization) in the app to regenerate the artifact.


In [9]:
if pipeline is not None:
    # Select only numeric ML features from df_cs_test (drop non-numeric / label cols)
    EXCLUDE = {"dtp_bracket", "date_fully_paid", "due_date",
               "school_year", "student_id_pseudonimized", "category_name", "description"}
    X_infer = df_cs_test.drop(columns=[c for c in EXCLUDE if c in df_cs_test.columns])

    df_preds = run_batch_inference(
        input_source=X_infer,
        model_dir=MODEL_DIR,
        batch_size=1024,
        return_proba=True,
    )
    display(df_preds.head(10))
else:
    df_preds = None
    print("Skipping inference -- no valid InferencePipeline loaded.")


Skipping inference -- no valid InferencePipeline loaded.


In [10]:
if df_preds is not None:
    print("Predicted label distribution:")
    display(df_preds["predicted_label"].value_counts().rename("count").to_frame())
    n = len(df_preds)
    mk = df_preds["model_key"].iloc[0]
    ts = df_preds["run_timestamp"].iloc[0]
    print()
    print("Total rows scored:", n)
    print("Model key        :", mk)
    print("Run timestamp    :", ts)
